In [ ]:
import subprocess, os

REPO_URL = "https://github.com/safety-research/legibility.git"
WORK_DIR = "/workspace/18-4-2026"

# Clone or pull latest
if not os.path.exists(os.path.join(WORK_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, WORK_DIR], check=True)
else:
    subprocess.run(["git", "-C", WORK_DIR, "pull"], check=True)

# Pull LFS files (.eval logs)
subprocess.run(["git", "-C", WORK_DIR, "lfs", "pull"], check=True)

# Install dependencies
subprocess.run(["pip", "install", "-q", "-r", os.path.join(WORK_DIR, "requirements.txt")], check=True)

# Set working directory to notebooks/ so Path.cwd().parent resolves to repo root
os.chdir(os.path.join(WORK_DIR, "notebooks"))

# NB2: Reader R2 Activation Extraction (H200 GPU)

Load R2 (Llama-3.1-70B-Instruct, 4-bit quantized) on H200 and extract activations
during C2 crossfill. This enables Experiment D: reader-side activation analysis.

**Outputs:**
- `activations/R2_last_token/` -- last-token acts at every 4th layer
- `activations/R2_cot_boundary/` -- activation at the prefill/generation boundary

**GPU time:** ~4 hours

**Note:** Uses `BitsAndBytesConfig(load_in_4bit=True)` (~40GB VRAM).
Within-model comparison (success vs. failure) remains valid despite quantization.

In [ ]:
import sys
import json
import gc
import torch
import numpy as np
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from phase2_utils import (
    load_phase1_results, join_cots_with_labels, load_model,
    extract_activations, save_activations, load_activations,
    get_default_layer_indices, print_phase1_summary,
    LOCAL_MODELS, ACTIVATIONS_DIR,
)

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
print_phase1_summary()

# For reader analysis, we need CoTs paired with their C2 outcome.
# Load all classified CoTs (all generators, all labels except FILTERED).
all_cots = join_cots_with_labels()
print(f"Total classified CoTs with text: {len(all_cots)}")

# Build R2 crossfill prompt format
r2_inputs = []
r2_labels = []  # 1 = C2 success (reader answered correctly), 0 = C2 failure
r2_metadata = []

results = load_phase1_results()
for rec in all_cots:
    # Get R2's C2 result for this CoT
    c2_results = rec.get('c2_results', {})
    r2_correct = c2_results.get('R2', None)
    if r2_correct is None:
        continue
    
    # Build crossfill input: question + CoT reasoning
    crossfill_input = (
        f"{rec['input']}\n\n"
        f"Here is a reasoning trace from another model:\n"
        f"<think>\n{rec['cot_text']}\n</think>\n\n"
        f"Based on the above reasoning, what is the answer?"
    )
    
    r2_inputs.append(crossfill_input)
    r2_labels.append(int(r2_correct))
    r2_metadata.append({
        'sample_id': rec['sample_id'],
        'generator_id': rec['generator_id'],
        'epoch': rec['epoch'],
        'label': rec['label'],
        'r2_correct': r2_correct,
    })

print(f"R2 crossfill inputs: {len(r2_inputs)}")
print(f"R2 C2 success rate: {sum(r2_labels)/len(r2_labels):.1%}")

# Check by legibility class
from collections import Counter
for label in ['ANSWER_LEAKED', 'REASONING_LEGIBLE', 'ILLEGIBLE']:
    subset = [m for m in r2_metadata if m['label'] == label]
    if subset:
        success = sum(1 for m in subset if m['r2_correct'])
        print(f"  {label}: {len(subset)} samples, R2 success={success/len(subset):.1%}")

In [ ]:
# Check if all R2 outputs already exist (skip model load if so)
r2_outputs = ["R2_last_token", "R2_cot_boundary"]
r2_all_cached = all((ACTIVATIONS_DIR / name / "metadata.json").exists() for name in r2_outputs)

if r2_all_cached:
    print("CACHED: All R2 outputs exist, skipping R2 model load")
    model_r2 = tok_r2 = None
    n_layers_r2 = None
else:
    # Load R2 (Llama-3.1-70B-Instruct) with 4-bit quantization
    model_r2, tok_r2 = load_model("R2", quantize_4bit=True)
    n_layers_r2 = model_r2.config.num_hidden_layers
    print(f"R2 layers: {n_layers_r2}, hidden_dim: {model_r2.config.hidden_size}")

In [ ]:
output_dir = ACTIVATIONS_DIR / "R2_last_token"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    r2_last_token = load_activations(output_dir)
else:
    # Extract last-token activations at every 4th layer
    layer_indices = get_default_layer_indices(n_layers_r2, stride=4)
    print(f"Extracting at layers: {layer_indices}")

    r2_last_token = extract_activations(
        model_r2, tok_r2, r2_inputs,
        layer_indices=layer_indices,
        pooling="last_token",
        batch_size=1,
        max_length=16384,
    )

    # Verify shapes
    for layer_idx, acts in r2_last_token.items():
        assert acts.shape[0] == len(r2_inputs)
        assert not np.any(np.isnan(acts))
        assert not np.any(np.isinf(acts))
        print(f"  Layer {layer_idx}: shape={acts.shape}, norm={np.linalg.norm(acts, axis=1).mean():.2f}")

    metadata_r2 = {
        "model": "R2",
        "hf_id": LOCAL_MODELS["R2"]["hf_id"],
        "pooling": "last_token",
        "quantization": "4bit_nf4",
        "n_samples": len(r2_inputs),
        "layer_indices": layer_indices,
        "sample_metadata": r2_metadata,
        "labels": r2_labels,
    }
    save_activations(r2_last_token, output_dir, metadata=metadata_r2)

In [ ]:
output_dir = ACTIVATIONS_DIR / "R2_cot_boundary"
if (output_dir / "metadata.json").exists():
    print(f"CACHED: {output_dir} already exists, skipping extraction")
    r2_boundary = load_activations(output_dir)
else:
    # Extract activation at the CoT/generation boundary
    # This is the position just after the </think> tag, where the reader
    # transitions from reading the CoT to generating its answer.
    from tqdm import tqdm

    layer_indices = get_default_layer_indices(n_layers_r2, stride=4)
    boundary_activations = {idx: [] for idx in layer_indices}

    for i in tqdm(range(len(r2_inputs)), desc="Extracting boundary activations"):
        text = r2_inputs[i]
        inputs = tok_r2(
            text, return_tensors="pt", truncation=True, max_length=16384
        ).to(next(model_r2.parameters()).device)
        
        # Find </think> boundary token position
        think_end_tokens = tok_r2.encode("</think>", add_special_tokens=False)
        input_ids = inputs["input_ids"][0].tolist()
        boundary_pos = None
        for j in range(len(input_ids) - len(think_end_tokens) + 1):
            if input_ids[j:j + len(think_end_tokens)] == think_end_tokens:
                boundary_pos = j + len(think_end_tokens)
                break
        
        if boundary_pos is None:
            # Fallback: use position at 90% of sequence (near end of CoT)
            boundary_pos = int(len(input_ids) * 0.9)
        
        boundary_pos = min(boundary_pos, len(input_ids) - 1)
        
        with torch.no_grad():
            outputs = model_r2(**inputs, output_hidden_states=True)
        
        for layer_idx in layer_indices:
            if layer_idx < len(outputs.hidden_states):
                act = outputs.hidden_states[layer_idx][0, boundary_pos, :].cpu().float().numpy()
                boundary_activations[layer_idx].append(act)
        
        del outputs
        torch.cuda.empty_cache()

    # Stack into arrays
    r2_boundary = {
        idx: np.stack(acts, axis=0)
        for idx, acts in boundary_activations.items()
        if acts
    }

    for layer_idx, acts in r2_boundary.items():
        print(f"  Layer {layer_idx}: shape={acts.shape}")

    metadata_r2_boundary = {
        "model": "R2",
        "hf_id": LOCAL_MODELS["R2"]["hf_id"],
        "pooling": "cot_boundary_position",
        "quantization": "4bit_nf4",
        "n_samples": len(r2_inputs),
        "layer_indices": layer_indices,
        "sample_metadata": r2_metadata,
        "labels": r2_labels,
    }
    save_activations(r2_boundary, output_dir, metadata=metadata_r2_boundary)

In [ ]:
# Cleanup
if model_r2 is not None:
    del model_r2, tok_r2
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("NB2 complete.")

In [ ]:
# Summary of saved files
for subdir in ['R2_last_token', 'R2_cot_boundary']:
    path = ACTIVATIONS_DIR / subdir
    if path.exists():
        total_size = sum(f.stat().st_size for f in path.rglob('*') if f.is_file())
        print(f"  {subdir}: {total_size / 1e9:.2f} GB")
    else:
        print(f"  {subdir}: NOT FOUND")